# 🧪 Lab 4 — Unidad 3: Merge Sort

**Universidad de Talca — Curso de Algoritmos y Estructuras de Datos**

| Aspecto | Detalle |
|--------|--------|
| **Profesor** | PhD. César Astudillo |
| **Unidad** | Unidad 3: Ordenamiento |
| **Tema** | Merge Sort — Divide & Conquer |
| **Duración** | 100 minutos (2 bloques de 50 min) |
| **Fecha** | 2026-05 |

---

## 📋 Instrucciones Generales

- Trabaja de forma **individual**.
- Ejecuta cada celda antes de pasar a la siguiente.
- El verificador automático te dirá cuántos casos pasan. Apunta a 100%.
- Al terminar un bloque, el ayudante revisará tu avance antes de continuar.
- **No modifiques** las celdas de Setup ni de verificación automática.

---

## Setup (NO MODIFICAR)

In [ ]:
# Setup del laboratorio — ejecutar primero
import sys, random, time, timeit, math
import matplotlib.pyplot as plt
import numpy as np

print("✅ Imports correctos")
print("🐍 Python", sys.version.split()[0])

# ── Implementaciones de referencia (usadas en los verificadores) ──────────

def _merge_ref(izq, der):
    """Merge de referencia (correcto y estable)."""
    resultado = []
    i = j = 0
    while i < len(izq) and j < len(der):
        if izq[i] <= der[j]: resultado.append(izq[i]); i += 1
        else:                 resultado.append(der[j]); j += 1
    resultado.extend(izq[i:]); resultado.extend(der[j:])
    return resultado

def _merge_sort_ref(lista):
    """Merge Sort de referencia."""
    if len(lista) <= 1: return lista[:]
    m = len(lista) // 2
    return _merge_ref(_merge_sort_ref(lista[:m]), _merge_sort_ref(lista[m:]))

def _insertion_ref(a):
    a = a[:]
    for i in range(1, len(a)):
        c = a[i]; j = i - 1
        while j >= 0 and a[j] > c: a[j+1] = a[j]; j -= 1
        a[j+1] = c
    return a

print("✅ Implementaciones de referencia cargadas")

---
# 🔵 BLOQUE 1 — La función Merge y Merge Sort básico (50 minutos)

> Al terminar este bloque, levanta la mano para que el ayudante revise tu progreso.

## PARTE 1A: Trazar Merge a mano (10 minutos)

Traza la fusión de las siguientes dos listas ordenadas **sin ejecutar código**:

**izq** = `[2, 5, 7, 10]`  
**der** = `[1, 3, 8, 9]`

Completa la tabla indicando en cada paso qué elemento se toma y de qué lista:

| Paso | izq[i] | der[j] | ¿Cuál tomamos? | Resultado parcial |
|------|--------|--------|----------------|-------------------|
| 1 | 2 | 1 | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |
| 6 | | | | |
| 7 | | | | |
| 8 | | | | |

**¿Cuántas comparaciones se necesitaron?** ___  
**¿Cuál es el máximo posible para dos listas de 4 elementos?** ___

### Tu Respuesta (edita esta celda)

_[Completa la tabla aquí]_

Comparaciones: ___  
Máximo posible: ___

In [ ]:
# Verificación del trazado manual
izq = [2, 5, 7, 10]
der = [1, 3, 8, 9]

print(f"izq = {izq}")
print(f"der = {der}")
print()
resultado = []; i = j = 0; comparaciones = 0
while i < len(izq) and j < len(der):
    comparaciones += 1
    if izq[i] <= der[j]:
        print(f"  Paso {comparaciones}: izq[{i}]={izq[i]} <= der[{j}]={der[j]} → tomo {izq[i]}")
        resultado.append(izq[i]); i += 1
    else:
        print(f"  Paso {comparaciones}: izq[{i}]={izq[i]} >  der[{j}]={der[j]} → tomo {der[j]}")
        resultado.append(der[j]); j += 1
    print(f"           Resultado parcial: {resultado}")
resultado.extend(izq[i:]); resultado.extend(der[j:])
if izq[i:]: print(f"  Copia resto de izq: {izq[i:]}")
if der[j:]: print(f"  Copia resto de der: {der[j:]}")
print(f"\nResultado final: {resultado}")
print(f"Comparaciones:   {comparaciones} (máximo: {len(izq)+len(der)-1})")

## PARTE 1B: Implementar la función Merge (15 minutos)

In [ ]:
def mi_merge(izq: list, der: list) -> tuple:
    """
    Fusiona dos listas ordenadas en una sola lista ordenada.

    Debe ser:
      - Correcto: la lista resultante está ordenada
      - Estable: elementos iguales mantienen su orden relativo original
      - Eficiente: O(n) donde n = len(izq) + len(der)

    Parámetros:
        izq (list): lista izquierda ya ordenada
        der (list): lista derecha ya ordenada

    Retorna:
        tuple: (lista_fusionada, n_comparaciones)
    """
    # Tu código aquí
    pass

In [ ]:
# Suite de Tests — mi_merge
def verificar_merge(fn):
    casos = [
        ([1, 3, 5], [2, 4, 6],   [1,2,3,4,5,6],  "Intercaladas perfectas"),
        ([1, 2, 3], [4, 5, 6],   [1,2,3,4,5,6],  "izq completamente menor"),
        ([4, 5, 6], [1, 2, 3],   [1,2,3,4,5,6],  "der completamente menor"),
        ([],        [1, 2, 3],   [1, 2, 3],       "izq vacía"),
        ([1, 2, 3], [],          [1, 2, 3],       "der vacía"),
        ([],        [],          [],              "Ambas vacías"),
        ([1, 1, 2], [1, 2, 3],   [1,1,1,2,2,3],  "Con repetidos"),
        ([5],       [5],         [5, 5],          "Iguales de 1 elemento"),
    ]
    # Test de estabilidad: tuplas (valor, orden_original)
    # Con izq=[(2,1),(2,3)] y der=[(2,2),(2,4)], el resultado estable es [(2,1),(2,2),(2,3),(2,4)]
    # Pero mi_merge trabaja con enteros, así que verificamos con una variante

    aprobados = 0
    for izq, der, esperado, desc in casos:
        try:
            resultado, cmp = fn(izq[:], der[:])
            ok = resultado == esperado
            # Verificar que retorna comparaciones (entero >= 0)
            ok_cmp = isinstance(cmp, int) and cmp >= 0
            # Verificar cota de comparaciones
            max_cmp = max(len(izq) + len(der) - 1, 0)
            ok_bound = cmp <= max_cmp
            if ok and ok_cmp and ok_bound:
                print(f"  ✅ {desc} | cmp={cmp}")
                aprobados += 1
            else:
                print(f"  ❌ {desc}")
                if not ok:      print(f"     Esperado: {esperado} | Obtenido: {resultado}")
                if not ok_cmp:  print(f"     comparaciones debe ser int>=0, obtuvo: {cmp}")
                if not ok_bound:print(f"     cmp={cmp} supera el máximo {max_cmp}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")

    # Test de estabilidad
    try:
        # Lista de tuplas (valor, id_original) — el id debe preservarse dentro de iguales
        izq_t = [(2, 0), (4, 2)]
        der_t = [(2, 1), (4, 3)]
        # Para esto necesitamos una versión genérica de merge — verificamos con enteros
        # Verificamos indirectamente: izq=[2,2,4] der=[2,3,4] → resultado[0] y [1] deben ser los 2 de izq primero
        izq_s = [2, 2, 4]; der_s = [2, 3, 4]
        res_s, _ = fn(izq_s, der_s)
        esperado_s = [2, 2, 2, 3, 4, 4]
        if res_s == esperado_s:
            print("  ✅ Resultado correcto con duplicados")
            aprobados += 1
        else:
            print(f"  ❌ Resultado con duplicados | Esperado: {esperado_s} | Obtenido: {res_s}")
    except Exception as e:
        print(f"  💥 Test duplicados — Error: {e}")

    total = len(casos) + 1
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == total else f'⚠️  {aprobados}/{total} casos correctos'}")

verificar_merge(mi_merge)

## PARTE 1C: Implementar Merge Sort completo (20 minutos)

In [ ]:
class MiMergeSort:
    """
    Implementación propia de Merge Sort.

    API:
        sort(lista)   → lista ordenada
        comparaciones → número de comparaciones realizadas
        niveles       → profundidad máxima de recursión alcanzada
    """

    def __init__(self):
        self.comparaciones = 0
        self.niveles       = 0

    def sort(self, lista: list) -> list:
        """
        Ordena lista con Merge Sort. Actualiza self.comparaciones y self.niveles.

        Parámetros:
            lista (list): lista de elementos comparables
        Retorna:
            list: nueva lista ordenada (no modifica la original)
        """
        self.comparaciones = 0
        self.niveles       = 0
        # Tu código aquí
        pass

In [ ]:
# Suite de Tests — MiMergeSort
def verificar_merge_sort(clase):
    import math, random
    casos = [
        ([5, 2, 8, 1, 9, 3, 7, 4],  "Lista general n=8"),
        ([],                          "Lista vacía"),
        ([1],                         "Un elemento"),
        ([2, 1],                      "Dos elementos invertidos"),
        ([1, 2, 3, 4, 5],            "Ya ordenada (mejor caso)"),
        ([5, 4, 3, 2, 1],            "Invertida (peor caso)"),
        ([3, 3, 3, 3],               "Todos iguales"),
        (random.sample(range(200), 50), "Aleatoria n=50"),
        (random.sample(range(1000), 200), "Aleatoria n=200"),
    ]
    aprobados = 0
    for lista, desc in casos:
        try:
            inst      = clase()
            resultado = inst.sort(lista[:])
            esperado  = sorted(lista)
            ok_orden  = resultado == esperado
            ok_no_mod = True  # Verificar que lista original no fue modificada
            original  = lista[:]
            inst.sort(lista)  # Re-llamar con la misma lista
            ok_no_mod = lista == original

            # Verificar niveles: para n>=2 debe ser ceil(log2(n))
            ok_niveles = True
            n = len(lista)
            if n >= 2:
                inst2 = clase()
                inst2.sort(lista)
                esperado_niveles = math.ceil(math.log2(n))
                ok_niveles = inst2.niveles == esperado_niveles

            if ok_orden and ok_no_mod and ok_niveles:
                print(f"  ✅ {desc} | cmp={inst.comparaciones}, niveles={inst.niveles}")
                aprobados += 1
            else:
                print(f"  ❌ {desc}")
                if not ok_orden:   print(f"     Orden incorrecto")
                if not ok_no_mod:  print(f"     Modificó la lista original")
                if not ok_niveles: print(f"     Niveles: esperado {esperado_niveles}, obtuvo {inst2.niveles}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")

    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados==len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_merge_sort(MiMergeSort)

## PARTE 1D: Comparación de comparaciones vs teoría (5 minutos)

In [ ]:
# Verifica que tus comparaciones sean consistentes con O(n log n)
import math, random

print(f"{'n':>6} | {'Cmp reales':>12} | {'n·log₂n':>10} | {'Ratio':>8} | {'¿OK?':>6}")
print("-" * 52)

ns = [8, 16, 32, 64, 128, 256, 512]
for n in ns:
    datos = random.sample(range(n * 5), n)
    inst  = MiMergeSort()
    if inst.sort(datos) is None:
        print(f"{n:>6} | (implementación pendiente)")
        continue
    cmp      = inst.comparaciones
    teorico  = n * math.log2(n)
    ratio    = cmp / teorico if teorico > 0 else 0
    # Ratio esperado entre 0.5 y 2.0 para una implementación correcta
    ok = 0.5 <= ratio <= 2.0
    print(f"{n:>6} | {cmp:>12} | {teorico:>10.1f} | {ratio:>8.3f} | {'✅' if ok else '⚠️ '}")

---
# 🟠 BLOQUE 2 — Merge Sort Avanzado (50 minutos)

> Al terminar este bloque, el ayudante revisará tu progreso final.

## PARTE 2A: Merge Sort con umbral (Timsort simplificado) (20 minutos)

El `sorted()` de Python usa **Timsort**, que combina Merge Sort con Insertion Sort.  
La idea: para subproblemas pequeños (n ≤ umbral), Insertion Sort es más rápido  
porque tiene menos overhead de función y mejor rendimiento de caché.

Implementa `merge_sort_umbral(lista, umbral)` que:
- Usa Insertion Sort cuando `len(lista) <= umbral`
- Usa Merge Sort recursivo para subproblemas más grandes

In [ ]:
def merge_sort_umbral(lista: list, umbral: int = 10) -> list:
    """
    Merge Sort híbrido con Insertion Sort para subproblemas pequeños.

    Parámetros:
        lista   (list): lista a ordenar
        umbral  (int):  tamaño máximo para usar Insertion Sort (default: 10)
    Retorna:
        list: nueva lista ordenada

    Complejidad: O(n log n) en general, con constante mejor que Merge Sort puro.
    """
    # Tu código aquí
    pass

In [ ]:
def verificar_umbral(fn):
    import random
    casos = [
        ([3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5], 5,  "n=11, umbral=5"),
        ([],                                  10, "Vacía"),
        ([42],                                10, "Un elemento"),
        (list(range(20, 0, -1)),              8,  "Invertida n=20, umbral=8"),
        (list(range(20)),                     8,  "Ya ordenada n=20, umbral=8"),
        (random.sample(range(500), 100),      15, "Aleatoria n=100, umbral=15"),
        (random.sample(range(500), 100),      1,  "Aleatoria n=100, umbral=1 (= Merge puro)"),
        (random.sample(range(500), 100),      100,"Aleatoria n=100, umbral=100 (= Insertion puro)"),
    ]
    aprobados = 0
    for lista, umbral, desc in casos:
        try:
            resultado = fn(lista[:], umbral)
            esperado  = sorted(lista)
            if resultado == esperado:
                print(f"  ✅ {desc}")
                aprobados += 1
            else:
                print(f"  ❌ {desc} | Resultado incorrecto")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados==len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_umbral(merge_sort_umbral)

In [ ]:
# Benchmark: encontrar el umbral óptimo
import timeit, random
import matplotlib.pyplot as plt

random.seed(42)
n      = 500
datos  = random.sample(range(n * 3), n)
reps   = 100
umbrales = [1, 2, 4, 8, 10, 12, 16, 20, 32, 50]
tiempos  = []

for u in umbrales:
    t = timeit.timeit(lambda: merge_sort_umbral(datos[:], u), number=reps) / reps * 1000
    tiempos.append(t)

# Referencia: merge sort puro y Python sorted
t_puro   = timeit.timeit(lambda: _merge_sort_ref(datos[:]), number=reps) / reps * 1000
t_sorted = timeit.timeit(lambda: sorted(datos),             number=reps) / reps * 1000

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(umbrales, tiempos, 'o-', color='#1565C0', linewidth=2, label='Merge+Insertion (tu impl.)')
ax.axhline(t_puro,   linestyle='--', color='#E53935', label=f'Merge puro ({t_puro:.3f}ms)')
ax.axhline(t_sorted, linestyle=':',  color='#43A047', label=f'Python sorted ({t_sorted:.3f}ms)')
idx_min = tiempos.index(min(tiempos))
ax.axvline(umbrales[idx_min], linestyle='-', color='orange', alpha=0.5,
           label=f'Umbral óptimo ≈ {umbrales[idx_min]}')
ax.set_xlabel('Umbral'); ax.set_ylabel('Tiempo (ms)')
ax.set_title(f'Impacto del umbral en Merge+Insertion Sort (n={n})')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Umbral óptimo encontrado: {umbrales[idx_min]}")
print(f"Speedup vs merge puro:    {t_puro/tiempos[idx_min]:.2f}×")

## PARTE 2B: Merge Sort bottom-up (iterativo) (15 minutos)

Merge Sort también puede implementarse **sin recursión** usando un enfoque bottom-up:

1. Empezamos con sublistas de tamaño 1 (ya ordenadas trivialmente)
2. Fusionamos pares de sublistas de tamaño 1 → sublistas de tamaño 2
3. Fusionamos pares de tamaño 2 → tamaño 4
4. Repetimos hasta que toda la lista esté fusionada

```
lista = [5, 2, 8, 1, 9, 3, 7, 4]

tamaño=1: [5][2][8][1][9][3][7][4]
tamaño=2: [2,5][1,8][3,9][4,7]
tamaño=4: [1,2,5,8][3,4,7,9]
tamaño=8: [1,2,3,4,5,7,8,9]
```

In [ ]:
def merge_sort_bottomup(lista: list) -> list:
    """
    Merge Sort iterativo (bottom-up) sin recursión.

    Parámetros:
        lista (list): lista a ordenar
    Retorna:
        list: nueva lista ordenada

    Complejidad:
        Temporal: O(n log n)
        Espacial: O(n) extra
        Ventaja: sin pila de recursión → O(log n) menos memoria que top-down
    """
    # Tu código aquí
    pass

In [ ]:
def verificar_bottomup(fn):
    import random
    casos = [
        ([5, 2, 8, 1, 9, 3, 7, 4],  "Ejemplo de la descripción n=8"),
        ([],                          "Vacía"),
        ([1],                         "Un elemento"),
        ([2, 1],                      "Dos elementos"),
        ([3, 3, 1, 1, 2, 2],         "Con repetidos"),
        (list(range(15, 0, -1)),      "Invertida n=15 (no potencia de 2)"),
        (list(range(16)),             "Ya ordenada n=16 (potencia de 2)"),
        (random.sample(range(300), 77), "Aleatoria n=77"),
        (random.sample(range(300), 128), "Aleatoria n=128"),
    ]
    aprobados = 0
    for lista, desc in casos:
        try:
            resultado = fn(lista[:])
            esperado  = sorted(lista)
            if resultado == esperado:
                print(f"  ✅ {desc}")
                aprobados += 1
            else:
                print(f"  ❌ {desc}")
                print(f"     Esperado: {esperado[:10]}..." if len(esperado)>10 else f"     Esperado: {esperado}")
                print(f"     Obtenido: {resultado[:10]}..." if len(resultado)>10 else f"     Obtenido: {resultado}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados==len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_bottomup(merge_sort_bottomup)

## PARTE 2C: Comparación final — todos los algoritmos (15 minutos)

In [ ]:
# Benchmark completo: todos los algoritmos vistos en la Unidad 3
import timeit, random, math
import matplotlib.pyplot as plt

def shell_sort_ref(lista):
    a = lista[:]; n = len(a)
    gaps = []
    g = 1
    while g < n // 3: g = 3 * g + 1; gaps.append(g)
    gaps = sorted(gaps, reverse=True) or [1]
    for gap in gaps:
        for i in range(gap, n):
            c = a[i]; j = i - gap
            while j >= 0 and a[j] > c: a[j+gap] = a[j]; j -= gap
            a[j+gap] = c
    return a

algoritmos = {
    'Insertion Sort': _insertion_ref,
    'Shell Sort':     shell_sort_ref,
    'Merge Sort':     _merge_sort_ref,
    'Python sorted':  sorted,
}

# Si el alumno implementó merge_sort_umbral y merge_sort_bottomup, agrégarlos
try:
    if merge_sort_umbral([3,1,2], 2) == [1,2,3]:
        algoritmos['Merge+Umbral (tuyo)'] = lambda x: merge_sort_umbral(x, 10)
except: pass
try:
    if merge_sort_bottomup([3,1,2]) == [1,2,3]:
        algoritmos['Merge BU (tuyo)'] = merge_sort_bottomup
except: pass

escenarios = {
    'Aleatorio':    lambda n: random.sample(range(n*3), n),
    'Casi ordenado':lambda n: sorted(random.sample(range(n*3), n))[:n//2] + random.sample(range(n*3), n//2),
    'Invertido':    lambda n: list(range(n, 0, -1)),
}

ns   = [100, 500, 1000, 2000, 5000]
cols = plt.cm.tab10.colors

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, (esc_nombre, esc_gen) in zip(axes, escenarios.items()):
    for idx, (nombre, fn) in enumerate(algoritmos.items()):
        tiempos = []
        for n in ns:
            datos = esc_gen(n)
            reps  = max(3, 200 // n)
            try:
                t = timeit.timeit(lambda: fn(datos[:]), number=reps) / reps * 1000
                tiempos.append(t)
            except:
                tiempos.append(None)
        ns_validos = [ns[i] for i,t in enumerate(tiempos) if t is not None]
        ts_validos = [t for t in tiempos if t is not None]
        if ts_validos:
            estilo = '-' if 'Sort' in nombre or 'sorted' in nombre else '--'
            ax.plot(ns_validos, ts_validos, marker='o', linestyle=estilo,
                    color=cols[idx % len(cols)], linewidth=1.8, label=nombre)
    ax.set_xlabel('n'); ax.set_ylabel('Tiempo (ms)')
    ax.set_title(f'Escenario: {esc_nombre}')
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.suptitle('Comparación de algoritmos de ordenamiento — Unidad 3',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## PARTE 2D: Preguntas de Análisis (10 minutos)

Responde las siguientes preguntas basándote en los resultados del benchmark:

### Pregunta 1

En el escenario **Aleatorio**, ¿a partir de qué valor de n Merge Sort supera a Shell Sort?  
¿Qué factor explica que Shell Sort sea competitivo para n pequeño?

_[Tu respuesta aquí]_

---

### Pregunta 2

En el escenario **Casi ordenado**, ¿Insertion Sort se comporta cercano a su mejor caso O(n)?  
¿Cómo se compara con Merge Sort en ese escenario?

_[Tu respuesta aquí]_

---

### Pregunta 3

¿Por qué `Python sorted()` (Timsort) es tan rápido en todos los escenarios?  
¿Qué ventajas tiene frente a tu implementación de Merge Sort?

_[Tu respuesta aquí]_

---

### Pregunta 4 (desafío)

Merge Sort usa O(n) memoria extra. Propón un escenario real donde esto sea un problema  
serio (p.ej. qué tipo de sistema, qué restricción de hardware). ¿Qué alternativa usarías?

_[Tu respuesta aquí]_

## 🔬 Zona de Experimentación

Sugerencias:
- Implementa Merge Sort para ordenar una lista enlazada (sin acceso aleatorio)
- ¿Qué pasa con el rendimiento si usas `lista + lista` en vez de `lista.extend()`?
- Implementa Natural Merge Sort que detecta "runs" ya ordenados en la entrada

In [ ]:
# Espacio libre para experimentar

In [ ]:
# Espacio libre para experimentar